In [1]:
import re
import warnings
import urllib.request

import nltk
import optuna
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from corus import load_lenta

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
SEED = 42

d:\vscode_projects\itmo_dl_nlp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Загрузка, предобработка

In [2]:
# Используем urllib для скачивания данных (если файл отсутствует)
data_url = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
local_filename = "lenta-ru-news.csv.gz"

try:
    with open(local_filename, "rb"):
        pass
except FileNotFoundError:
    print("Downloading dataset...")
    urllib.request.urlretrieve(data_url, local_filename)
    print("Download completed.")

Download completed.


In [3]:
# Загружаем данные с помощью Corus
raw_records = load_lenta(local_filename)
df = pd.DataFrame(raw_records)
df.columns = ['url', 'title', 'text', 'topic', 'tags', 'date']
df = df[['title', 'text', 'topic']]
df = df.sample(n=20000, random_state=SEED).reset_index(drop=True)
df.head()

,title,text,topic
0,EgyptAir объявила о подорожании билетов,Египетский перевозчик EgyptAir сообщил о возмо...,Путешествия
1,Глава Красногорского района Подмосковья ушел в...,Глава Красногорского района Московской области...,Россия
2,Милонов предложил запретить россиянам сидеть в...,Депутат Виталий Милонов внес в Госдуму законоп...,Россия
3,Женщинам в детородном возрасте разрешили посещ...,Верховный суд Индии разрешил женщинам в фертил...,Мир
4,Россиянам пообещали дешевый хлеб,Россиянам не стоит бояться роста цен на хлеб —...,Экономика


In [4]:
# Фильтрация редких классов: оставляем только топики с не менее чем 1 примерами
topic_freq = df['topic'].value_counts()
popular_topics = topic_freq[topic_freq > 1].index
df = df[df['topic'].isin(popular_topics)].reset_index(drop=True)
df['topic'].value_counts()

topic
Россия               4380
Мир                  3702
Экономика            2154
Спорт                1687
Наука и техника      1494
Культура             1446
Бывший СССР          1393
Интернет и СМИ       1261
Из жизни              725
Дом                   570
Силовые структуры     539
Ценности              215
Бизнес                183
Путешествия           181
69-я параллель         34
Крым                   15
Культпросвет            7
Библиотека              5
                        5
Легпром                 4
Name: count, dtype: int64

In [5]:
# Загрузка инструментов для обработки текста
nltk.download('stopwords')
russian_stop = set(stopwords.words("russian"))
stemmer = SnowballStemmer("russian")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ivann\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [6]:
def clean_text(raw_text: str) -> str:
    """
    Функция очистки текста с использованием стемминга:
    - Приведение к нижнему регистру,
    - Удаление HTML-тегов,
    - Сохранение русских слов и удаление пунктуации,
    - Токенизация, удаление стоп-слов и стемминг.
    """
    txt = raw_text.lower()
    txt = re.sub(r"<.*?>", " ", txt)
    txt = re.sub(r"[^а-яё\s]", " ", txt)
    tokens = [w for w in txt.split() if w not in russian_stop]
    stemmed = [stemmer.stem(token) for token in tokens]
    return " ".join(stemmed)

Базовая предобработка, включающая очистку (удаление HTML-тегов (не проверял, но такое может быть), перевод в нижний регистр, фильтрация символов и удаление стоп-слов) и стемминг, была выбрана для быстрой обработки данных и уменьшения вычислительных затрат. Базовая обработка для удаления шума из данных и снижения размерности признакового пространства. Сохраняю только русские слова для снижения размера признаков (наверняка слова английские встречаются слишком редко).

In [7]:
df["full_text"] = (df["title"] + " " + df["text"]).apply(clean_text)
df["full_text"].head()

0    объяв подорожан билет египетск перевозчик сооб...
1    глав красногорск район подмосков ушел отставк ...
2    милон предлож запрет россиян сидет соцсет рабо...
3    женщин детородн возраст разреш посеща индуистс...
4    россиян пообеща дешев хлеб россиян сто боя рос...
Name: full_text, dtype: object

In [8]:
# Разбиваем данные на train/validation/test (60/20/20) с сохранением пропорций классов
X = df["full_text"]
y = df["topic"]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, train_size=0.6, stratify=y, random_state=SEED)
X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)

In [9]:
# --- Dummy baseline ---
dummy_pipe = Pipeline([
    ("vec", CountVectorizer()),
    ("dummy", DummyClassifier(strategy="most_frequent", random_state=SEED))
])
dummy_pipe.fit(X_train, y_train)
dummy_preds = dummy_pipe.predict(X_valid)
print("Dummy baseline report:")
print(classification_report(y_valid, dummy_preds))

Dummy baseline report:
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         1
   69-я параллель       0.00      0.00      0.00         7
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.00      0.00      0.00        37
      Бывший СССР       0.00      0.00      0.00       279
              Дом       0.00      0.00      0.00       114
         Из жизни       0.00      0.00      0.00       145
   Интернет и СМИ       0.00      0.00      0.00       252
             Крым       0.00      0.00      0.00         3
    Культпросвет        0.00      0.00      0.00         1
         Культура       0.00      0.00      0.00       289
          Легпром       0.00      0.00      0.00         1
              Мир       0.00      0.00      0.00       740
  Наука и техника       0.00      0.00      0.00       299
      Путешествия       0.00      0.00      0.00        36
           Россия       0.22    

#### Пайплайны с LogisticRegression
Ниже приведены два варианта: с использованием CountVectorizer и TfidfVectorizer.
Сначала базовые модели без оптимизации.

In [10]:
# Базовая модель с CountVectorizer
pipe_count = Pipeline([
    ("vec", CountVectorizer()),
    ("clf", LogisticRegression(random_state=SEED, max_iter=1000))
])
pipe_count.fit(X_train, y_train)
count_preds = pipe_count.predict(X_valid)
print("Report (CountVectorizer + LogReg):")
print(classification_report(y_valid, count_preds))

Report (CountVectorizer + LogReg):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         1
   69-я параллель       1.00      0.29      0.44         7
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.45      0.14      0.21        37
      Бывший СССР       0.79      0.70      0.74       279
              Дом       0.80      0.74      0.77       114
         Из жизни       0.55      0.50      0.52       145
   Интернет и СМИ       0.72      0.65      0.69       252
             Крым       0.00      0.00      0.00         3
    Культпросвет        0.00      0.00      0.00         1
         Культура       0.87      0.84      0.86       289
          Легпром       0.00      0.00      0.00         1
              Мир       0.73      0.78      0.76       740
  Наука и техника       0.77      0.81      0.79       299
      Путешествия       0.74      0.47      0.58        36
           Россия   

In [11]:
# Базовая модель с TfidfVectorizer
pipe_tfidf = Pipeline([
    ("vec", TfidfVectorizer()),
    ("clf", LogisticRegression(random_state=SEED, max_iter=1000))
])
pipe_tfidf.fit(X_train, y_train)
tfidf_preds = pipe_tfidf.predict(X_valid)
print("Report (TfidfVectorizer + LogReg):")
print(classification_report(y_valid, tfidf_preds))

Report (TfidfVectorizer + LogReg):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         1
   69-я параллель       0.00      0.00      0.00         7
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.00      0.00      0.00        37
      Бывший СССР       0.80      0.62      0.70       279
              Дом       0.91      0.65      0.76       114
         Из жизни       0.73      0.37      0.49       145
   Интернет и СМИ       0.79      0.60      0.68       252
             Крым       0.00      0.00      0.00         3
    Культпросвет        0.00      0.00      0.00         1
         Культура       0.84      0.85      0.85       289
          Легпром       0.00      0.00      0.00         1
              Мир       0.72      0.85      0.78       740
  Наука и техника       0.77      0.84      0.80       299
      Путешествия       0.86      0.17      0.28        36
           Россия   

#### Оптимизация гиперпараметров с использованием Optuna

Здесь подбираю одновременно параметры векторизаторов и LogisticRegression. Для каждого варианта (CountVectorizer и TfidfVectorizer) определим отдельную функцию-цель.

In [12]:
def objective_count(trial: optuna.Trial) -> float:
    # Параметры векторизатора
    max_df = trial.suggest_float("max_df", 0.7, 0.9)
    min_df = trial.suggest_float("min_df", 0.003, 0.01)
    ngram_low = 1
    ngram_high = trial.suggest_int("ngram_high", 1, 2)
    
    # Параметры классификатора
    C_val = trial.suggest_float("C", 0.02, 5.0, log=True)
    solver_opt = trial.suggest_categorical("solver", ["liblinear", "lbfgs"])
    if solver_opt == "liblinear":
        penalty_opt = trial.suggest_categorical("penalty", ["l1", "l2"])
    else:
        penalty_opt = "l2"
    
    pipe = Pipeline([
        ("vec", CountVectorizer(max_df=max_df, min_df=min_df, ngram_range=(ngram_low, ngram_high))),
        ("clf", LogisticRegression(C=C_val, solver=solver_opt, penalty=penalty_opt,
                                     random_state=SEED, max_iter=1000))
    ])
    
    # Используем кросс-валидацию для оценки 
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy", n_jobs=-1)
    return scores.mean()

In [13]:
study_count = optuna.create_study(direction="maximize")
study_count.optimize(objective_count, n_trials=20)

print("Best parameters for CountVectorizer pipeline:")
print(study_count.best_params)
print("Best CV accuracy:", study_count.best_value)

[I 2025-03-04 19:00:51,449] A new study created in memory with name: no-name-64b429a7-2e1e-42af-8269-faa631975384
[I 2025-03-04 19:00:58,809] Trial 0 finished with value: 0.72125 and parameters: {'max_df': 0.7922232246978904, 'min_df': 0.006617845682197357, 'ngram_high': 2, 'C': 0.8719404820593639, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.72125.
[I 2025-03-04 19:01:06,790] Trial 1 finished with value: 0.6891666666666667 and parameters: {'max_df': 0.8467993123900375, 'min_df': 0.0072255606537866045, 'ngram_high': 2, 'C': 3.5447204142431135, 'solver': 'liblinear', 'penalty': 'l1'}. Best is trial 0 with value: 0.72125.
[I 2025-03-04 19:01:14,048] Trial 2 finished with value: 0.7335 and parameters: {'max_df': 0.7311769647904626, 'min_df': 0.0069022034047101, 'ngram_high': 2, 'C': 0.13723485189673734, 'solver': 'lbfgs'}. Best is trial 2 with value: 0.7335.
[I 2025-03-04 19:01:18,320] Trial 3 finished with value: 0.7406666666666667 and parameters: {'max_df': 0.7046871991097339, 'min

Best parameters for CountVectorizer pipeline:
{'max_df': 0.7896235752218985, 'min_df': 0.003082070165286759, 'ngram_high': 1, 'C': 0.02404496778124899, 'solver': 'liblinear', 'penalty': 'l2'}
Best CV accuracy: 0.7590833333333332


In [14]:
def objective_tfidf(trial: optuna.Trial) -> float:
    # Параметры векторизатора TF-IDF
    max_df = trial.suggest_float("max_df", 0.7, 0.9)
    min_df = trial.suggest_float("min_df", 0.003, 0.01)
    ngram_low = 1
    ngram_high = trial.suggest_int("ngram_high", 1, 2)
    
    # Параметры классификатора
    C_val = trial.suggest_float("C", 0.02, 5.0, log=True)
    solver_opt = trial.suggest_categorical("solver", ["liblinear", "lbfgs"])
    if solver_opt == "liblinear":
        penalty_opt = trial.suggest_categorical("penalty", ["l1", "l2"])
    else:
        penalty_opt = "l2"
    
    pipe = Pipeline([
        ("vec", TfidfVectorizer(max_df=max_df, min_df=min_df, ngram_range=(ngram_low, ngram_high))),
        ("clf", LogisticRegression(C=C_val, solver=solver_opt, penalty=penalty_opt,
                                     random_state=SEED, max_iter=1000))
    ])
    
    scores = cross_val_score(pipe, X_train, y_train, cv=3, scoring="accuracy", n_jobs=-1)
    return scores.mean()

In [15]:
study_tfidf = optuna.create_study(direction="maximize")
study_tfidf.optimize(objective_tfidf, n_trials=20)

print("Best parameters for TfidfVectorizer pipeline:")
print(study_tfidf.best_params)
print("Best CV accuracy:", study_tfidf.best_value)

[I 2025-03-04 19:02:45,530] A new study created in memory with name: no-name-c748f098-b61b-4155-8f4f-347f3ff4da7b
[I 2025-03-04 19:02:48,120] Trial 0 finished with value: 0.69175 and parameters: {'max_df': 0.8822488506629819, 'min_df': 0.004822245205045698, 'ngram_high': 1, 'C': 0.21281513492460716, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.69175.
[I 2025-03-04 19:02:50,936] Trial 1 finished with value: 0.7480833333333333 and parameters: {'max_df': 0.7488567327865228, 'min_df': 0.006045197053551926, 'ngram_high': 1, 'C': 1.1923668195753323, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.7480833333333333.
[I 2025-03-04 19:02:52,898] Trial 2 finished with value: 0.6881666666666667 and parameters: {'max_df': 0.891465186705644, 'min_df': 0.0035412234756748632, 'ngram_high': 1, 'C': 0.38098635183128865, 'solver': 'liblinear', 'penalty': 'l1'}. Best is trial 1 with value: 0.7480833333333333.
[I 2025-03-04 19:02:54,815] Trial 3 finished with value: 0.45599999999999996 and parameter

Best parameters for TfidfVectorizer pipeline:
{'max_df': 0.7495013624760425, 'min_df': 0.006024148657174658, 'ngram_high': 1, 'C': 1.721178881621881, 'solver': 'liblinear', 'penalty': 'l2'}
Best CV accuracy: 0.7505833333333335


#### Финальная оценка на тестовой выборке.

Строим финальные модели с оптимальными параметрами и оцениваем их на отложенной выборке.
Для каждого пайплайна сначала выводим оценку на валидационной выборке, затем на тестовой.


In [16]:
# Модель для CountVectorizer
best_count_pipe = Pipeline([
    ("vec", CountVectorizer(
        max_df=study_count.best_params["max_df"],
        min_df=study_count.best_params["min_df"],
        ngram_range=(1, study_count.best_params["ngram_high"])
    )),
    ("clf", LogisticRegression(
        C=study_count.best_params["C"],
        solver=study_count.best_params["solver"],
        penalty=study_count.best_params["penalty"],
        random_state=SEED,
        max_iter=1000
    ))
])

In [ ]:
best_count_pipe.fit(X_train, y_train)
val_preds_count = best_count_pipe.predict(X_valid)
test_preds_count = best_count_pipe.predict(X_test)
print("Final Report on Validation Set (CountVectorizer):")
print(classification_report(y_valid, val_preds_count))

Final Report on Validation Set (CountVectorizer):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         1
   69-я параллель       0.00      0.00      0.00         7
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.33      0.05      0.09        37
      Бывший СССР       0.79      0.65      0.71       279
              Дом       0.86      0.70      0.77       114
         Из жизни       0.63      0.49      0.55       145
   Интернет и СМИ       0.77      0.62      0.69       252
             Крым       0.00      0.00      0.00         3
    Культпросвет        0.00      0.00      0.00         1
         Культура       0.86      0.84      0.85       289
          Легпром       0.00      0.00      0.00         1
              Мир       0.72      0.81      0.76       740
  Наука и техника       0.77      0.80      0.79       299
      Путешествия       0.80      0.33      0.47        36
     

In [21]:
print("Final Report on Test Set (CountVectorizer):")
print(classification_report(y_test, test_preds_count))

Final Report on Test Set (CountVectorizer):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         1
   69-я параллель       1.00      0.14      0.25         7
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.50      0.06      0.10        36
      Бывший СССР       0.77      0.69      0.73       278
              Дом       0.87      0.72      0.79       114
         Из жизни       0.53      0.38      0.44       145
   Интернет и СМИ       0.71      0.62      0.66       252
             Крым       0.00      0.00      0.00         3
    Культпросвет        0.00      0.00      0.00         2
         Культура       0.82      0.82      0.82       289
          Легпром       0.00      0.00      0.00         1
              Мир       0.73      0.82      0.77       741
  Наука и техника       0.78      0.78      0.78       299
      Путешествия       0.75      0.33      0.46        36
           

In [18]:
# Модель для TfidfVectorizer
best_tfidf_pipe = Pipeline([
    ("vec", TfidfVectorizer(
        max_df=study_tfidf.best_params["max_df"],
        min_df=study_tfidf.best_params["min_df"],
        ngram_range=(1, study_tfidf.best_params["ngram_high"])
    )),
    ("clf", LogisticRegression(
        C=study_tfidf.best_params["C"],
        solver=study_tfidf.best_params["solver"],
        penalty=study_tfidf.best_params["penalty"],
        random_state=SEED,
        max_iter=1000
    ))
])

In [ ]:
best_tfidf_pipe.fit(X_train, y_train)
val_preds_tfidf = best_tfidf_pipe.predict(X_valid)
test_preds_tfidf = best_tfidf_pipe.predict(X_test)
print("Final Report on Validation Set (TfidfVectorizer):")
print(classification_report(y_valid, val_preds_tfidf))

Final Report on Validation Set (TfidfVectorizer):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         1
   69-я параллель       0.00      0.00      0.00         7
       Библиотека       0.00      0.00      0.00         1
           Бизнес       1.00      0.03      0.05        37
      Бывший СССР       0.78      0.62      0.69       279
              Дом       0.88      0.65      0.75       114
         Из жизни       0.61      0.43      0.50       145
   Интернет и СМИ       0.76      0.58      0.66       252
             Крым       0.00      0.00      0.00         3
    Культпросвет        0.00      0.00      0.00         1
         Культура       0.84      0.84      0.84       289
          Легпром       0.00      0.00      0.00         1
              Мир       0.72      0.82      0.77       740
  Наука и техника       0.76      0.82      0.79       299
      Путешествия       0.87      0.36      0.51        36
     

In [20]:
print("Final Report on Test Set (TfidfVectorizer):")
print(classification_report(y_test, test_preds_tfidf))

Final Report on Test Set (TfidfVectorizer):
                   precision    recall  f1-score   support

                        0.00      0.00      0.00         1
   69-я параллель       0.00      0.00      0.00         7
       Библиотека       0.00      0.00      0.00         1
           Бизнес       0.00      0.00      0.00        36
      Бывший СССР       0.75      0.63      0.68       278
              Дом       0.85      0.68      0.76       114
         Из жизни       0.62      0.33      0.43       145
   Интернет и СМИ       0.72      0.63      0.67       252
             Крым       0.00      0.00      0.00         3
    Культпросвет        0.00      0.00      0.00         2
         Культура       0.81      0.81      0.81       289
          Легпром       0.00      0.00      0.00         1
              Мир       0.71      0.82      0.76       741
  Наука и техника       0.77      0.79      0.78       299
      Путешествия       0.79      0.31      0.44        36
           

Чуть лучше отработал CountVectorizer - {'max_df': 0.7896235752218985, 'min_df': 0.003082070165286759, 'ngram_high': 1, 'C': 0.02404496778124899, 'solver': 'liblinear', 'penalty': 'l2'}
- Best CV accuracy: 0.7590833333333332
- validation: accuracy 0.76
- test: accuracy 0.76